In [1]:
%load_ext autoreload
%autoreload 2
import os

os.chdir("..")
print(os.getcwd())

/Users/tplas/repos/aether


In [2]:
import re

import pandas as pd
import requests
import wandb

In [3]:
def pprint(dictionary):
    for k, v in dictionary.items():
        print(k, ":", v)

In [4]:
os.makedirs("outputs/alignment", exist_ok=True)

# Table prep functions

In [5]:
def format_mean_sem(mean, sem, decimals=3, scale=1.0, math_mode=True):
    """Format a single mean/SEM pair as a LaTeX string, with optional scaling."""
    if pd.isna(mean):
        return "--"
    m = mean * scale
    if pd.isna(sem):
        s = f"{m:.{decimals}f}"
    else:
        s = f"{m:.{decimals}f} $\\pm$ {sem * scale:.{decimals}f}"
    return f"{s}" if math_mode else s


def add_mean_sem_columns(df, metrics, math_mode=True):
    """
    metrics: dict mapping output_col_name -> dict with:
        mean_col, sem_col, decimals (int), scale (float, default 1.0)
    """
    df = df.copy()
    for out_col, spec in metrics.items():
        df[out_col] = [
            format_mean_sem(
                m,
                s,
                decimals=spec.get("decimals", 3),
                scale=spec.get("scale", 1.0),
                math_mode=math_mode,
            )
            for m, s in zip(df[spec["mean_col"]], df[spec["sem_col"]])
        ]
    return df


def get_latex(df, columns=None, escape=False):
    if columns is not None:
        df = df[columns]
    else:
        columns = list(df.columns)

    latex_table = df.to_latex(
        index=False,
        columns=columns,
        escape=escape,  # False: let \pm and $ pass through untouched
    )
    print(latex_table)

    payload = {
        "formula": latex_table,
        "fsize": "54px",
        "fcolor": "000000",
        "mode": "0",
        "out": "1",
        "remhost": "quicklatex.com",
        "preamble": r"\usepackage{booktabs}\usepackage{amsmath}",
    }
    response = requests.post("https://quicklatex.com/latex3.f", data=payload)
    print(response.text)
    return latex_table


def parse_modality(x, modalities):
    for k, v in modalities.items():
        if k in x:
            return v

# WANDB API parsing

In [7]:
# Connect to wandb api
api = wandb.Api()
entity = "aether_xai"

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/tplas/.netrc


In [8]:
project = "satbird-usa-summer_alignment"
results_df_path = "outputs/alignment/results_s2bms.csv"

## csv log

In [9]:
df = None

# Separate concept columns

In [10]:
os.getcwd()

'/Users/tplas/repos/aether'

In [11]:
import json

# Collect columns into 3 categories per version
versions = {"v4": {}}

for k, v in versions.items():
    with open(f"data/satbird-USA-summer/concept_captions/{k}.json", "r") as f:
        data = json.load(f)

    v["in_dist_valid"] = []
    v["in_dist_unvalid"] = []

    for d in data:
        if "test" in d["col"]:
            name = d["col"].replace("test-", "test_dyn_k_index_")
            name = f"{name.replace('aux_', '')}_{'max' if d['is_max'] else 'min'}"
            v["out_dist"].append(name)
        else:
            name = f"{d['col'].replace('aux_', 'test_dyn_k_index_')}_{'max' if d['is_max'] else 'min'}"
            if d.get("val_av", True):
                v["in_dist_valid"].append(name)
            else:
                v["in_dist_unvalid"].append(name)

In [12]:
versions["v4"]

{'in_dist_valid': ['test_dyn_k_index_bio_2_max',
  'test_dyn_k_index_bio_3_max',
  'test_dyn_k_index_bio_5_max',
  'test_dyn_k_index_bio_6_max',
  'test_dyn_k_index_bio_11_max',
  'test_dyn_k_index_bio_12_max',
  'test_dyn_k_index_bio_13_max',
  'test_dyn_k_index_bio_14_max',
  'test_dyn_k_index_bio_15_max',
  'test_dyn_k_index_bio_16_max',
  'test_dyn_k_index_bio_17_max',
  'test_dyn_k_index_bio_18_max',
  'test_dyn_k_index_bio_19_max',
  'test_dyn_k_index_clyppt_max',
  'test_dyn_k_index_phihox_max',
  'test_dyn_k_index_sltppt_max',
  'test_dyn_k_index_sndppt_max',
  'test_dyn_k_index_dw_water_max',
  'test_dyn_k_index_dw_trees_max',
  'test_dyn_k_index_dw_grass_max',
  'test_dyn_k_index_dw_flooded_vegetation_max',
  'test_dyn_k_index_dw_crops_max',
  'test_dyn_k_index_dw_shrub_and_scrub_max',
  'test_dyn_k_index_dw_built_max',
  'test_dyn_k_index_dw_bare_max'],
 'in_dist_unvalid': ['test_dyn_k_index_bdticm_max',
  'test_dyn_k_index_cecsol_max',
  'test_dyn_k_index_orcdrc_max',
  'te

## fetching

In [13]:
def log_concept_avr(run, cols, name):
    metrics = dict(run.summary)
    matched_metrics = [metrics[c] for c in cols]
    value = sum(matched_metrics) / len(matched_metrics)
    run.summary.update({name: value})
    print(name, value)


def flatten(d: dict, parent_key: str = "", sep: str = ".") -> dict:
    """Flatten a nested config dict (e.g. a Hydra config) into dot-separated keys."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

In [14]:
projects = ["satbird-usa-summer_alignment"]

In [15]:
api = wandb.Api()
for project in projects:
    runs_iterator = api.runs(f"{entity}/{project}", order="+created_at")
    run_list = []

    for i, run in enumerate(runs_iterator):
        if run.state != "finished":
            print(f"{run.id} is not finished.")
            continue
        concept_v = run.config["data"]["caption_builder"]["concepts_fname"].replace(".json", "")
        if concept_v == "${version}":
            concept_v = run.summary["experiment"]
            match = re.search(r"(?<=_)v\d+", concept_v)
            concept_v = match.group() if match else None

        if run.summary.get("test_avr_top-dyn_k_index_in_dist_valid") is None:
            try:
                in_dist_valid = versions[concept_v]["in_dist_valid"]
                in_dist_unvalid = versions[concept_v]["in_dist_unvalid"]

                log_concept_avr(run, in_dist_valid, "test_avr_top-dyn_k_index_in_dist_valid")
                log_concept_avr(run, in_dist_unvalid, "test_avr_top-dyn_k_index_in_dist_unvalid")
            except Exception as e:
                if concept_v == "v8":
                    print(f"v8 issue {run.id}: {e}")
                else:
                    print(f"{run.id}: {e}")

        if df is not None:
            if run.id in list(df.run_id):
                print(
                    f'{run.summary["experiment"]} with seed={run.config["seed"]} already in results.'
                )
                continue

        temp = dict(run.summary)
        config_dict = flatten(run.config)
        captures = {}
        captures["experiment"] = run.summary["experiment"]
        captures["run_id"] = run.id
        for k, v in temp.items():
            # if "test" not in k and "best" not in k:
            #     captures[f"best_{k}"] = v
            # else:
            #     captures[k] = v
            if "test" in k and "dyn_k_index" in k:
                captures[k] = v

        captures = {**captures, **config_dict}
        del captures["tags"]
        run_list.append(captures)
        print(f'{run.summary["experiment"]} with seed={run.config["seed"]} logged.')

aef_b512 with seed=12345 logged.
aef_b512 with seed=654 logged.
aef_b512 with seed=404 logged.
geoclip_baseline with seed=12345 logged.
baseline_clip_ViT-B-32_div_2000 with seed=12345 logged.
baseline_clip_ViT-B-32_div_10000 with seed=12345 logged.
baseline_clip_ViT-B-32_stretch_2_98 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_4c_div_2000 with seed=12345 logged.
baseline_clip_ViT-B-16_div_2000 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_4c_div_10000 with seed=12345 logged.
baseline_clip_ViT-B-16_div_10000 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_4c_stretch_2_98 with seed=12345 logged.
baseline_clip_ViT-B-16_stretch_2_98 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_rgb_div_2000 with seed=12345 logged.
baseline_clip_ViT-L-14_div_2000 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_rgb_div_10000 with seed=12345 logged.
baseline_dofaclip_ViT-B-16_rgb_stretch_2_98 with seed=12345 logged.
baseline_clip_ViT-L-14_div_10000 with seed=12345 logged.
baseline_dofac

In [36]:
list_aef = []
list_other = []

for r in run_list:
    # if r['experiment'].startswith('aef'):
    #     list_aef.append(r)
    # elif r['experiment'] in ['higher', 'lower']:
    #     continue
    # else:
    #     list_other.append(r)
    if r["experiment"].startswith("baseline") or r["experiment"].startswith("geoclip"):
        list_other.append(r)
    else:
        list_aef.append(r)
print(f"len(list_aef)={len(list_aef)}, len(list_other)={len(list_other)}")
df_dict = {}
df_dict["aef"] = pd.DataFrame(list_aef)
if len(list_other) > 0:
    df_dict["baseline"] = pd.DataFrame(list_other)

import re

for k, v in df_dict.items():
    test_cols = [c for c in v.columns if "test" in c and "dyn_k_index" in c]

    dict_cols = {}

    dict_cols["bioclim"] = [c for c in test_cols if "_bio_" in c]
    dict_cols["dw_all"] = [c for c in test_cols if "_dw_" in c]
    soil_vars = ["bdticm", "cecsol", "clyppt", "orcdrc", "phihox", "sltppt", "sndppt"]
    dict_cols["soil"] = [c for c in test_cols if any([s in c for s in soil_vars])]

    cols_groups = []
    for k2, v2 in dict_cols.items():
        new_col = f"test_avr_top-dyn_k_index_aggr_{k2}"
        df_dict[k][new_col] = df_dict[k][v2].mean(axis=1)
        cols_groups.append(new_col)
    df_dict[k]["group_av"] = df_dict[k][cols_groups].mean(axis=1)

    cols_drop = []

    for c in v.columns:
        if any([isinstance(x, list) for x in v[c]]):
            cols_drop.append(c)
        elif v[c].nunique() <= 1:
            cols_drop.append(c)

    df_dict[k].drop(columns=cols_drop, inplace=True)

len(list_aef)=25, len(list_other)=55


In [37]:
df_dict["aef"].sort_values(
    by="test_avr_top-dyn_k_index_in_dist_valid", ascending=False, inplace=True
)
df_dict["aef"]

,experiment,run_id,test_avr_top-dyn_k_index,test_avr_top-dyn_k_index_in_dist_unvalid,test_avr_top-dyn_k_index_in_dist_valid,test_dyn_k_index_bdticm_max,test_dyn_k_index_bio_11_max,test_dyn_k_index_bio_12_max,test_dyn_k_index_bio_13_max,test_dyn_k_index_bio_14_max,...,data.batch_size,seed,model.loss_fn.sigma,model.optimizer.lr,callbacks.early_stopping.patience,optimizer.lr,test_avr_top-dyn_k_index_aggr_bioclim,test_avr_top-dyn_k_index_aggr_dw_all,test_avr_top-dyn_k_index_aggr_soil,group_av
6,s0.1,se7as899,0.223961,0.053268,0.251272,-0.056046,-0.115607,-0.016685,0.222010,-0.024855,...,512,404,0.1,0.0010,15,0.0010,0.096590,0.565022,0.022001,0.227871
8,s0.1,idoavsbt,0.230320,0.108035,0.249886,0.180328,-0.150087,-0.066390,0.206968,-0.015725,...,512,654,0.1,0.0010,15,0.0010,0.090817,0.561926,0.063047,0.238597
21,s0.1_b256,9p0mogew,0.210935,0.068511,0.233723,-0.029084,-0.191656,0.046575,0.165186,0.027643,...,256,654,0.1,0.0010,10,0.0010,0.071992,0.559165,0.021247,0.217468
9,s0.1,k3ql24nf,0.206227,0.049029,0.231379,-0.065034,-0.224203,-0.002226,0.162679,0.002536,...,512,12345,0.1,0.0010,15,0.0010,0.065638,0.566973,0.003505,0.212039
12,s0.1_b128,tw7a1e67,0.214512,0.118710,0.229840,0.194708,-0.140420,-0.055545,0.154322,-0.006594,...,128,12345,0.1,0.0010,10,0.0010,0.043941,0.557311,0.090544,0.230599
11,s0.1_b128,wmk5ovru,0.212822,0.110988,0.229115,0.190215,-0.235159,-0.059160,0.222845,-0.002029,...,128,404,0.1,0.0010,10,0.0010,0.050464,0.549010,0.082102,0.227192
20,s0.1_b256,1t2hhok8,0.206034,0.079877,0.226220,0.052704,-0.183600,-0.016685,0.172706,-0.011160,...,256,404,0.1,0.0010,10,0.0010,0.048372,0.557892,0.046448,0.217571
16,s0.1_b256,lfqwfyjy,0.203335,0.091302,0.221260,0.137188,-0.257072,0.005004,0.241230,0.007101,...,256,12345,0.1,0.0010,10,0.0010,0.049610,0.550102,0.042982,0.214231
13,s0.1_b128,uxze7167,0.180508,0.093355,0.194453,0.090452,-0.268673,-0.020300,0.109197,0.007101,...,128,654,0.1,0.0010,10,0.0010,0.004583,0.547324,0.035606,0.195838
2,aef_b512,6njr0un8,0.178261,0.085134,0.193161,0.029336,-0.184245,-0.067294,0.111704,0.007101,...,512,404,0.5,0.0010,30,0.0010,0.016932,0.532437,0.022501,0.190624


# AEF

In [38]:
best_aef_run_config = df_dict["aef"][df_dict["aef"]["experiment"] == "s0.1"]
assert len(best_aef_run_config) == 3, "There should be only one best AEF run."

In [43]:
# best_aef_run_config["model.geo_encoder._target_"]  # = df_dict['aef']

# BASELINE

In [44]:
df_dict['aef']

,experiment,run_id,test_avr_top-dyn_k_index,test_avr_top-dyn_k_index_in_dist_unvalid,test_avr_top-dyn_k_index_in_dist_valid,test_dyn_k_index_bdticm_max,test_dyn_k_index_bio_11_max,test_dyn_k_index_bio_12_max,test_dyn_k_index_bio_13_max,test_dyn_k_index_bio_14_max,...,data.batch_size,seed,model.loss_fn.sigma,model.optimizer.lr,callbacks.early_stopping.patience,optimizer.lr,test_avr_top-dyn_k_index_aggr_bioclim,test_avr_top-dyn_k_index_aggr_dw_all,test_avr_top-dyn_k_index_aggr_soil,group_av
6,s0.1,se7as899,0.223961,0.053268,0.251272,-0.056046,-0.115607,-0.016685,0.222010,-0.024855,...,512,404,0.1,0.0010,15,0.0010,0.096590,0.565022,0.022001,0.227871
8,s0.1,idoavsbt,0.230320,0.108035,0.249886,0.180328,-0.150087,-0.066390,0.206968,-0.015725,...,512,654,0.1,0.0010,15,0.0010,0.090817,0.561926,0.063047,0.238597
21,s0.1_b256,9p0mogew,0.210935,0.068511,0.233723,-0.029084,-0.191656,0.046575,0.165186,0.027643,...,256,654,0.1,0.0010,10,0.0010,0.071992,0.559165,0.021247,0.217468
9,s0.1,k3ql24nf,0.206227,0.049029,0.231379,-0.065034,-0.224203,-0.002226,0.162679,0.002536,...,512,12345,0.1,0.0010,15,0.0010,0.065638,0.566973,0.003505,0.212039
12,s0.1_b128,tw7a1e67,0.214512,0.118710,0.229840,0.194708,-0.140420,-0.055545,0.154322,-0.006594,...,128,12345,0.1,0.0010,10,0.0010,0.043941,0.557311,0.090544,0.230599
11,s0.1_b128,wmk5ovru,0.212822,0.110988,0.229115,0.190215,-0.235159,-0.059160,0.222845,-0.002029,...,128,404,0.1,0.0010,10,0.0010,0.050464,0.549010,0.082102,0.227192
20,s0.1_b256,1t2hhok8,0.206034,0.079877,0.226220,0.052704,-0.183600,-0.016685,0.172706,-0.011160,...,256,404,0.1,0.0010,10,0.0010,0.048372,0.557892,0.046448,0.217571
16,s0.1_b256,lfqwfyjy,0.203335,0.091302,0.221260,0.137188,-0.257072,0.005004,0.241230,0.007101,...,256,12345,0.1,0.0010,10,0.0010,0.049610,0.550102,0.042982,0.214231
13,s0.1_b128,uxze7167,0.180508,0.093355,0.194453,0.090452,-0.268673,-0.020300,0.109197,0.007101,...,128,654,0.1,0.0010,10,0.0010,0.004583,0.547324,0.035606,0.195838
2,aef_b512,6njr0un8,0.178261,0.085134,0.193161,0.029336,-0.184245,-0.067294,0.111704,0.007101,...,512,404,0.5,0.0010,30,0.0010,0.016932,0.532437,0.022501,0.190624


In [45]:
df_dict["baseline"][
    df_dict["baseline"]["model._target_"] == "src.models.text_alignment_model.TextAlignmentModel"
]

,experiment,run_id,test_avr_top-dyn_k_index,test_avr_top-dyn_k_index_in_dist_unvalid,test_avr_top-dyn_k_index_in_dist_valid,test_dyn_k_index_bdticm_max,test_dyn_k_index_bio_11_max,test_dyn_k_index_bio_12_max,test_dyn_k_index_bio_13_max,test_dyn_k_index_bio_14_max,...,model.text_encoder.use_geoclip_projector,model/params/total,model/params/trainable,data.dataset.modalities.s2.channels,data.dataset.modalities.s2.preprocessing,model.model_name,test_avr_top-dyn_k_index_aggr_bioclim,test_avr_top-dyn_k_index_aggr_dw_all,test_avr_top-dyn_k_index_aggr_soil,group_av
0,geoclip_baseline,40bcf0b1,0.059914,0.028504,0.064940,0.157859,0.010713,0.091761,-0.013643,0.039056,...,True,124634625,124634625,NaN,NaN,NaN,0.045843,0.105499,0.027437,0.059593
1,baseline_clip_ViT-B-32_div_2000,kz7s0en0,0.105995,-0.006042,0.123921,-0.032679,0.054215,0.032115,-0.026178,-0.006594,...,False,151277313,151277313,rgb,div_2000,NaN,0.064611,0.224919,0.029950,0.106493
2,baseline_clip_ViT-B-32_div_10000,vgr3q9hn,0.124246,0.015138,0.141703,0.069780,-0.060181,0.068264,-0.016986,0.036774,...,False,151277313,151277313,rgb,div_10000,NaN,0.057218,0.298763,0.024349,0.126777
3,baseline_clip_ViT-B-32_stretch_2_98,h5x77ccx,0.121014,0.008053,0.139088,0.014057,-0.047614,0.024886,0.019783,-0.004312,...,False,151277313,151277313,rgb,stretch_2_98,NaN,0.056447,0.292573,0.020349,0.123123
5,baseline_clip_ViT-B-16_div_2000,yll7r4n9,0.117371,-0.000855,0.136287,0.016753,0.050993,0.054708,0.005576,0.000253,...,False,149620737,149620737,rgb,div_2000,NaN,0.071038,0.248852,0.034371,0.118087
7,baseline_clip_ViT-B-16_div_10000,b99vjybm,0.125987,0.027890,0.141683,0.025741,-0.046969,0.061938,0.012262,-0.018007,...,False,149620737,149620737,rgb,div_10000,NaN,0.072933,0.297568,0.003914,0.124805
9,baseline_clip_ViT-B-16_stretch_2_98,fbjsrhm1,0.133316,0.033175,0.149339,0.050008,-0.039557,0.024886,0.065743,0.004818,...,False,149620737,149620737,rgb,stretch_2_98,NaN,0.082325,0.300125,0.013547,0.131999
11,baseline_clip_ViT-L-14_div_2000,j37g69pi,0.151446,-0.007881,0.176938,-0.028185,0.170223,0.033019,0.025632,0.025361,...,False,427616513,427616513,rgb,div_2000,NaN,0.118322,0.279306,0.048571,0.148733
14,baseline_clip_ViT-L-14_div_10000,3ow5ydmu,0.138611,0.023036,0.157103,-0.002121,-0.066948,0.041153,0.039002,0.039056,...,False,427616513,427616513,rgb,div_10000,NaN,0.079838,0.332920,-0.002067,0.136897
16,baseline_clip_ViT-L-14_stretch_2_98,6qe804se,0.156498,0.044292,0.174451,0.067983,-0.033435,0.005004,0.039002,0.018513,...,False,427616513,427616513,rgb,stretch_2_98,NaN,0.085132,0.356995,0.031256,0.157794


In [57]:
save_table = True
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c
    not in ["experiment", "run_id", "model/params/total", "model/params/trainable", "group_av"]
]
save_tables = False
criterion_metric = "test_avr_top-dyn_k_index_in_dist_valid"
metrics_table = [
    "test_avr_top-dyn_k_index_aggr_bioclim",
    "test_avr_top-dyn_k_index_aggr_dw_all",
    "test_avr_top-dyn_k_index_aggr_soil",
]
metrics_use = metrics_table + [criterion_metric]
columns_name_map = {
    "name_plot": "Model",
    "test_avr_top-dyn_k_index_aggr_bioclim": "Biocl.",
    "test_avr_top-dyn_k_index_aggr_dw_all": "DW LC",
    "test_avr_top-dyn_k_index_aggr_soil": "Soil",
    "average": "Av.",
    "test_avr_top-dyn_k_index_in_dist_valid": "Av. (sub)",
    "data.dataset.modalities.s2.preprocessing": "S2 preprocess.",
    "model.model_name": "Model name",
    "data.dataset.modalities.s2.channels": "Chan.",
}
model_name_map = {
    "AEF_ablations": {
        "AEF labelled, AEF proj.": "AEF, labelled, AEF proj., sep.",
        "AEF unlabelled, AEF proj.": "AEF, unlabelled, AEF proj., sep.",
        "AEF unlabelled, text proj.": "AEF, unlabelled, text proj., sep.",
        "AEF unlabelled, both proj.": "AEF, unlabelled, both proj., sep.",
        "AEF unlabelled, AEF proj., mixed": "AEF, unlabelled, AEF proj., mix.",
        "AEF labelled, AEF proj., mixed": "AEF, labelled, AEF proj., mix.",
    },
    "main": {
        "AEF": "AEF, separate captions",
        "TextAlignmentModel": "CLIP",
        "ClipImgEncoder": "CLIP",
        "DOFACLIP_model": "DOFA-CLIP",
        "GeoClipCoordinateEncoder": "GeoCLIP",
        "GeoCLIP": "GeoCLIP",
        "GeoRSCLIP_model": "GeoRSCLIP",
        "RemoteCLIP_model": "RemoteCLIP",
        "SkyCLIP_model": "SkyCLIP",
    },
}
assert len(best_aef_run_config) == 3, "There should be only one best AEF run."

best_aef_run_config["name_plot"] = "AEF"
runs_sel = (
    best_aef_run_config.groupby("name_plot")
    .agg({k: ["mean", "sem"] for k in metrics_use})
    .reset_index()
)
runs_sel["name_plot"] = "AEF"
df_best_runs = runs_sel.copy()  # Ensure the best_run has the same columns as df_best_runs

for metric in metrics_use:
    mean_col = (metric, "mean")
    sem_col = (metric, "sem")
    tmp = [
        format_mean_sem(m, s, decimals=2, scale=1.0, math_mode=True)
        for m, s in zip(df_best_runs[mean_col], df_best_runs[sem_col])
    ]
    df_best_runs.drop(columns=[mean_col, sem_col], inplace=True)
    if metric == "test_avr_top-dyn_k_index_aggr_topo":
        continue
    df_best_runs[metric] = tmp

df_best_runs.columns = [c[0] if isinstance(c, tuple) else c for c in df_best_runs.columns]


## get best baseline model for each combination of model and geo_encoder, and compare to best AEF model
runs_df = df_dict["baseline"].copy()

cols_params = [
    c
    for c in runs_df.columns
    if "index" not in c
    and c not in ["experiment", "run_id", "model/params/total", "model/params/trainable"]
]
# cols_main = ['model._target_', 'model.geo_encoder._target_']
# unique_combinations = runs_df[cols_main].drop_duplicates()
# assert False
runs_df["name_plot"] = runs_df["model._target_"].apply(lambda x: x.split(".")[-1].lstrip("build_"))
id_geoclip = runs_df['experiment'].str.startswith('geoclip')
runs_df.loc[id_geoclip, "name_plot"] = "GeoCLIP"
if "TextAlignmentModel" in runs_df["name_plot"].values:
    runs_df["name_plot"][runs_df["name_plot"] == "TextAlignmentModel"] = "CLIP"
for name_combi in runs_df["name_plot"].unique():
    runs_sel = runs_df[runs_df["name_plot"] == name_combi].copy()
    runs_sel["average"] = runs_sel[metrics_table].mean(axis=1)
    best_run = runs_sel.loc[runs_sel[(criterion_metric)].idxmax()]
    best_run = best_run[
        df_best_runs.columns
    ]  # Ensure the best_run has the same columns as df_best_runs
    assert all(
        best_run.index == df_best_runs.columns
    ), f"Column mismatch: {best_run.index} vs {df_best_runs.columns}"
    df_best_runs = pd.concat([df_best_runs, pd.DataFrame([best_run])], ignore_index=True)


## Select different model name maps for different tables
for name_map, model_map in model_name_map.items():
    if name_map != "main":
        continue
    df_best_runs_sel = df_best_runs.copy()
    df_best_runs_sel = df_best_runs_sel[
        df_best_runs_sel["name_plot"].isin(model_map.keys())
    ].copy()
    df_best_runs_sel["name_plot"] = df_best_runs_sel["name_plot"].map(model_map)
    df_best_runs_sel.rename(columns=columns_name_map, inplace=True)

    for c in df_best_runs_sel.columns:
        if c == "Model":
            continue
        df_best_runs_sel[c] = df_best_runs_sel[c].apply(
            lambda x: f"{x:.2f}" if isinstance(x, float) else x
        )

    def get_float_from_str(df, col_ignore=["Model"]):
        df_float = df.copy()
        for c in df_float.columns:
            if c in col_ignore:
                continue
            df_float[c] = df_float[c].apply(
                lambda x: float(x.split(" ")[0]) if isinstance(x, str) and " " in x else float(x)
            )
        return df_float

    df_best_runs_float = get_float_from_str(df_best_runs_sel, col_ignore=["Model"])
    inds_sorted = df_best_runs_float["Av. (sub)"].sort_values(ascending=True).index
    ## Put AEF last (index 0):
    inds_sorted = [i for i in inds_sorted if i != 0] + [0]
    df_best_runs_sel = df_best_runs_sel.loc[inds_sorted].reset_index(drop=True)
    df_best_runs_float = df_best_runs_float.loc[inds_sorted].reset_index(drop=True)

    ## best model per column:
    for c in df_best_runs_float.columns:
        if c == "Model":
            continue
        best_val = df_best_runs_float[c].max()
        inds_best = df_best_runs_float[df_best_runs_float[c] == best_val].index
        df_best_runs_sel.loc[inds_best, c] = df_best_runs_sel.loc[inds_best, c].apply(
            lambda x: f"\\textbf{{{x}}}" if isinstance(x, str) else f"\\textbf{{{x:.2f}}}"
        )

    if name_map == "main":
        caption = "\\textbf{Retrieval scores of the AlphaEarth (AEF) alignment models on the SatBird task.} Scores show the average retrieval $I_{\\text{top-}k_l}$ ($\\uparrow$) score for the bioclimatic, Dynamic World (DW) land cover, and soil groups, as well as the average across a subset (Av.\\ (sub)) of columns (used as validation criterion)."
    # elif name_map == 'AEF_ablations':
    #     caption = "\\textbf{Ablation study of the AlphaEarth alignment model on S2BMS.} The projection network (AEF-only, text-only, both), data (labelled, unlabelled), and caption strategy (separate, mixed) were varied."

    if save_table:
        df_best_runs_sel.to_latex(
            f"/Users/tplas/repos/ms_aether_biodiv/tables/alignment_satbird_summary_{name_map}.tex",
            index=False,
            escape=False,
            label=f"tab:align_satbird_summary_{name_map}",
            caption=caption,
        )

/var/folders/1n/bygyv77j7rj271kd_nj98h740000gn/T/ipykernel_87199/2482667367.py:93: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  runs_df["name_plot"][runs_df["name_plot"] == "TextAlignmentModel"] = "CLIP"


In [56]:
df_best_runs_sel

,Model,Biocl.,DW LC,Soil,Av. (sub)
0,GeoCLIP,0.05,0.11,0.03,0.06
1,RemoteCLIP,0.05,0.33,0.03,0.15
2,CLIP,0.12,0.28,0.05,0.18
3,DOFA-CLIP,0.04,0.45,\textbf{0.06},0.19
4,SkyCLIP,0.11,0.38,0.02,0.20
5,GeoRSCLIP,\textbf{0.15},0.42,0.03,0.23
6,"AEF, separate captions",0.08 $\pm$ 0.01,\textbf{0.56 $\pm$ 0.00},0.03 $\pm$ 0.02,\textbf{0.24 $\pm$ 0.01}


In [ ]:
df_dict["baseline"].sort_values(
    by="test_avr_top-dyn_k_index_in_dist_valid", ascending=False, inplace=True
)
df_dict["baseline"]

In [ ]:
soil_vars = ["bdticm", "cecsol", "clyppt", "orcdrc", "phihox", "sltppt", "sndppt"]
len(soil_vars)

In [52]:
runs_df

,experiment,run_id,test_avr_top-dyn_k_index,test_avr_top-dyn_k_index_in_dist_unvalid,test_avr_top-dyn_k_index_in_dist_valid,test_dyn_k_index_bdticm_max,test_dyn_k_index_bio_11_max,test_dyn_k_index_bio_12_max,test_dyn_k_index_bio_13_max,test_dyn_k_index_bio_14_max,...,model/params/total,model/params/trainable,data.dataset.modalities.s2.channels,data.dataset.modalities.s2.preprocessing,model.model_name,test_avr_top-dyn_k_index_aggr_bioclim,test_avr_top-dyn_k_index_aggr_dw_all,test_avr_top-dyn_k_index_aggr_soil,group_av,name_plot
0,geoclip_baseline,40bcf0b1,0.059914,0.028504,0.064940,0.157859,0.010713,0.091761,-0.013643,0.039056,...,124634625,124634625,NaN,NaN,NaN,0.045843,0.105499,0.027437,0.059593,GeoCLIP
1,baseline_clip_ViT-B-32_div_2000,kz7s0en0,0.105995,-0.006042,0.123921,-0.032679,0.054215,0.032115,-0.026178,-0.006594,...,151277313,151277313,rgb,div_2000,NaN,0.064611,0.224919,0.029950,0.106493,TextAlignmentModel
2,baseline_clip_ViT-B-32_div_10000,vgr3q9hn,0.124246,0.015138,0.141703,0.069780,-0.060181,0.068264,-0.016986,0.036774,...,151277313,151277313,rgb,div_10000,NaN,0.057218,0.298763,0.024349,0.126777,TextAlignmentModel
3,baseline_clip_ViT-B-32_stretch_2_98,h5x77ccx,0.121014,0.008053,0.139088,0.014057,-0.047614,0.024886,0.019783,-0.004312,...,151277313,151277313,rgb,stretch_2_98,NaN,0.056447,0.292573,0.020349,0.123123,TextAlignmentModel
4,baseline_dofaclip_ViT-B-16_4c_div_2000,et1kvnlf,0.131144,0.013743,0.149928,0.093148,-0.094983,0.110739,-0.002780,0.009383,...,245261058,245261058,4c,div_2000,ViT-B-16,0.064953,0.294120,0.044531,0.134535,DOFACLIP_model
5,baseline_clip_ViT-B-16_div_2000,yll7r4n9,0.117371,-0.000855,0.136287,0.016753,0.050993,0.054708,0.005576,0.000253,...,149620737,149620737,rgb,div_2000,NaN,0.071038,0.248852,0.034371,0.118087,TextAlignmentModel
6,baseline_dofaclip_ViT-B-16_4c_div_10000,ul904mrn,0.124829,0.015606,0.142304,0.061691,-0.100461,0.025789,0.002234,0.102967,...,245261058,245261058,4c,div_10000,ViT-B-16,0.051661,0.300492,0.034858,0.129004,DOFACLIP_model
7,baseline_clip_ViT-B-16_div_10000,b99vjybm,0.125987,0.027890,0.141683,0.025741,-0.046969,0.061938,0.012262,-0.018007,...,149620737,149620737,rgb,div_10000,NaN,0.072933,0.297568,0.003914,0.124805,TextAlignmentModel
8,baseline_dofaclip_ViT-B-16_4c_stretch_2_98,jttmfnux,0.148774,0.029415,0.167871,0.142580,-0.114962,0.059227,-0.027014,0.068729,...,245261058,245261058,4c,stretch_2_98,ViT-B-16,0.054039,0.358404,0.055185,0.155876,DOFACLIP_model
9,baseline_clip_ViT-B-16_stretch_2_98,fbjsrhm1,0.133316,0.033175,0.149339,0.050008,-0.039557,0.024886,0.065743,0.004818,...,149620737,149620737,rgb,stretch_2_98,NaN,0.082325,0.300125,0.013547,0.131999,TextAlignmentModel


## Previous:


In [ ]:
cols_of_interest = {
    "experiment": "Modality (best config.)",
    "test_avr_top-dyn_k_index_in_dist_valid": "In distribution (present)",
    "test_avr_top-dyn_k_index_in_dist_unvalid": "In distribution (absent)",
}

In [ ]:
sub_runs_df = runs_df[cols_of_interest.keys()]

grouped = sub_runs_df.groupby("experiment")

# mean and SEM in one go
summary_mean = grouped.mean(numeric_only=True)
summary_sem = grouped.sem(numeric_only=True)  # pandas has this built in: std/sqrt(n)

# also useful to know how many runs went into each SEM
summary_n = grouped.size().rename("n_runs")

# rename sem columns so they don't clash with mean columns
summary_sem = summary_sem.rename(columns={c: f"{c}_sem" for c in summary_sem.columns})

summary = summary_mean.join(summary_sem).join(summary_n)
summary = summary.sort_values("test_avr_top-dyn_k_index_in_dist_valid", ascending=False)
summary.reset_index(inplace=True)

In [ ]:
summary